# Set Up

## Mount Google Drive

Ignore if not using Google Collab:

In [1]:
from google.colab import drive

# mount google drive
drive.mount('/content/drive')
%cd /content/drive/My Drive
!git clone https://github.com/FranciscoLozCoding/cooling_with_code.git
%cd cooling_with_code
!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive
fatal: destination path 'cooling_with_code' already exists and is not an empty directory.
/content/drive/My Drive/cooling_with_code
Already up to date.


## Import Libraries

In [2]:
%%capture
%pip install shap
%pip install stackstac
%pip install pystac-client
%pip install planetary-computer
%pip install odc-stac
%pip install rioxarray
%pip install geopy
%pip install geopandas

In [3]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

#data science
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
from sklearn.feature_selection import RFECV
from sklearn.model_selection import GridSearchCV
import pickle
import shap

#other
import os

#custom tools
from tools.environment import VALID_SPLIT, RANDOM_STATE
from tools.feature_selection import get_feature_importance, plot_feature_importance_comparison
from tools.distribution import plot_target_var_distribution
from tools.preprocess import apply_preprocessing_mixed_buffers
from tools.explainer import Explainer
from tools.build_dataset import combine_buffer_datasets

In [4]:
train_combined, test_combined = combine_buffer_datasets('data/train', 'data/test')

Loading training data: data/train/50m_buffer_dataset.csv
Loading test data: data/test/50m_buffer_test_dataset.csv
Loading training data: data/train/100m_buffer_dataset.csv
Loading test data: data/test/100m_buffer_test_dataset.csv
Loading training data: data/train/150m_buffer_dataset.csv
Loading test data: data/test/150m_buffer_test_dataset.csv
Loading training data: data/train/300m_buffer_dataset.csv
Loading test data: data/test/300m_buffer_test_dataset.csv

Final Training Dataset Shape: (11229, 77)
Final Test Dataset Shape: (1040, 78)


In [5]:
train_combined, test_combined = apply_preprocessing_mixed_buffers(train_combined, test_combined)


Final Processed Training Dataset Shape: (11229, 164)
Final Processed Testing Dataset Shape: (1040, 165)


In [6]:
# Separate features and target
y = train_combined['UHI']
X = train_combined.drop('UHI', axis=1)

# Remove constant columns
X = X.loc[:, X.std() != 0]

#scale
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

#split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=VALID_SPLIT, random_state=RANDOM_STATE
)

#make into df
x_train_df = pd.DataFrame(X_train, columns=X.columns)
x_valid_df = pd.DataFrame(X_valid, columns=X.columns)

In [ ]:
# Base Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)

# Recursive Feature Elimination with Cross-Validation
rfe = RFECV(estimator=rf, step=1, cv=5, scoring="r2", n_jobs=-1)

# Fit to training data
rfe.fit(x_train_df, y_train)

# Get selected features
selected_features = x_train_df.loc[:, rfe.support_]

# Get names of selected features
selected_feature_names = x_train_df.columns[rfe.support_]

# Print selected feature names
print("Selected Features:", list(selected_feature_names))

In [8]:
# save the selected features so we don't have to run again
selected_feature_names=['50m_1NPCRI', '100m_Elevation_Wind_X', '150m_Traffic_Volume',
                        '150m_Elevation_Wind_Y', '150m_Humidity_NDVI', '150m_Traffic_NDBI',
                        '300m_SI', '300m_NPCRI', '300m_Coastal_Aerosol', '300m_Total_Building_Area_m2',
                        '300m_Building_Construction_Year', '300m_Ground_Elevation', '300m_Building_Wind_X',
                        '300m_Building_Wind_Y', '300m_Elevation_Wind_Y', '300m_BldgHeight_Count',
                        '300m_TotalBuildingArea_NDVI', '300m_Traffic_NDVI', '300m_Traffic_NDBI']

In [9]:
# select features
x_train_selected = pd.DataFrame(X_train, columns=x_train_df.columns)[selected_feature_names]
x_valid_selected = pd.DataFrame(X_valid, columns=x_train_df.columns)[selected_feature_names]

In [10]:
# Define hyperparameter grid
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False]
}

# Initialize model
rf_model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)

# Grid search with selected features
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

# Fit grid search to the **selected features only**
grid_search.fit(x_train_selected, y_train)

# Best parameters
print("Best Hyperparameters:", grid_search.best_params_)
print("Best R-squared Score:", grid_search.best_score_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best Hyperparameters: {'bootstrap': False, 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best R-squared Score: 0.9544851498323004


In [ ]:
# save the best params so we don't have to run again
best_hparams = {'bootstrap': False, 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

In [11]:
# best model, skip if model was already generated
model_pkl = 'models/mixed_buffers_RandomForestRegressor/mixed_buffers_RandomForestRegressor_model.pkl'
if not os.path.exists(model_pkl):
  best_rf_model = grid_search.best_estimator_
else:
  with open(model_pkl, 'rb') as f:
    best_rf_model = pickle.load(f)

In [12]:
# Make predictions on the training data
insample_predictions = best_rf_model.predict(x_train_selected)

# calculate R-squared score for in-sample predictions
print(f"In-Sample Evaluation:")

insample_r2 = r2_score(y_train, insample_predictions)
print(f"  300m buffer zone R-squared: {insample_r2}")

In-Sample Evaluation:
  300m buffer zone R-squared: 1.0


In [13]:
# Make predictions on the validation data
outsample_predictions = best_rf_model.predict(x_valid_selected)

# calculate R-squared score for in-sample predictions
print(f"Out-Sample Evaluation:")

outsample_r2 = r2_score(y_valid, outsample_predictions)
print(f"  300m buffer zone R-squared: {outsample_r2}")

Out-Sample Evaluation:
  300m buffer zone R-squared: 0.9640863723848668


In [14]:
# Save the model and scaler to files
with open('mixed_buffers_RandomForestRegressor_model.pkl', 'wb') as f:
    pickle.dump(best_rf_model, f)
with open('mixed_buffers_standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)